[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/15_calidad_regresion/15_calidad_regresion.ipynb)

# Módulo 15 · Calidad y regresión: cazar el cambio que empeora al agente sin que ningún dashboard lo vea

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 4 h 30 min (lección ≈ 2 h 45 min + proyecto ≈ 1 h 45 min) |
| **Servicios de Compose** | Opcional: `--profile core --profile metrics` (sección 9). Sin Docker todo funciona en **modo degradado** (telemetría en memoria, mismo esquema). Funciona en Colab |
| **RAM aproximada** | < 500 MB sin stack · ≈ 1,2 GB con `core` + `metrics` |
| **Prerrequisitos** | Módulo 14 (evaluación online: juez, muestreo, sesgos), 09 (métricas), 07 (TraceQL) y 13 (coste) |
| **Hilo conductor** | Alguien «mejora» el prompt de SupportOps para que sea más conciso. La latencia baja, el coste baja, los errores no se mueven… y el agente empieza a inventarse datos |

> ⚠️ **Honestidad sobre el laboratorio.** Las versiones de prompt son **simuladas** sobre el `mock` (`PromptVariantLLM`): `v2` hace que, en una fracción de los tickets que
> necesitan una herramienta, el modelo conteste directamente y se invente el dato. No es un modelo real, pero reproduce con exactitud el **patrón** de una regresión silenciosa
> y deja el *ground truth* en los datos para medir tus detectores. Las cifras cambian con modelos reales; el método no.

## Objetivos de aprendizaje

Al terminar esta lección podrás:

1. **Explicar** por qué una regresión de prompt/modelo es silenciosa y qué señales de telemetría la delatan (calidad, trayectoria, mezcla de herramientas).
2. **Construir** un dataset de regresión *desde trazas* (casos que fallaron, 👎, escalados y una muestra de los buenos), sin duplicados ni PII.
3. **Reproducir** (replay) el dataset contra dos versiones del agente y decidir con una **compuerta de release** con estadística apareada (McNemar, IC de la diferencia).
4. **Calcular** el poder estadístico: el tamaño de muestra necesario y la caída mínima detectable; entender el problema de «mirar los resultados antes de tiempo».
5. **Medir** el *drift* de entrada (PSI, KS) y de comportamiento (Jensen–Shannon sobre la mezcla de herramientas, huella de comportamiento) y detectar caídas con CUSUM.
6. **Diseñar** un despliegue en canario con decisión secuencial (SPRT) y calcular cuándo puedes promover o revertir.
7. **Detectar** una regresión de prompt **usando solo telemetría** (spans, eventos de evaluación) y localizar las trazas culpables con TraceQL.
8. **Comparar** versiones A/B del agente con guardarraíles de coste y latencia.

## 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 14) ¿Por qué la media de la muestra evaluada no es la calidad real y cómo se corrige?</summary>

Porque el muestreo es estratificado (más peso a errores y escalados). Se corrige con la **post-estratificación**: ponderar cada estrato por su tamaño en la población (`supportops_eval_population_total`).

</details>

<details>
<summary>2 · (Módulo 14) Un juez con sensibilidad 0,90 y especificidad 0,98 marca el 11 % de respuestas como malas. ¿Qué tasa real hay?</summary>

Rogan–Gladen: (0,11 + 0,98 − 1)/(0,90 + 0,98 − 1) ≈ 10,2 %. Al comparar versiones, el **mismo juez** sesga las dos igual: por eso las comparativas pareadas son más fiables que los valores absolutos.

</details>

<details>
<summary>3 · (Módulo 07) ¿Qué selector de TraceQL devuelve los `invoke_agent` sin ninguna llamada a herramienta?</summary>

Por atributos del span raíz: `{ name =~ "invoke_agent.*" && span.supportops.tool_calls = 0 }`. O estructural: trazas con `chat` pero sin hijos `execute_tool` (con `!>`/`count()` según versión de Tempo). Hoy lo usaremos para localizar la regresión.

</details>

<details>
<summary>4 · (Módulo 08/09, transferencia) Vas a etiquetar cada evaluación con la versión del agente (`v1`, `v2`, `v3-canary`…). ¿Cómo la guardas en una métrica de Prometheus y cómo en Loki? ¿Qué te preocupa a largo plazo?</summary>

En la **métrica** es una etiqueta de baja cardinalidad (`gen_ai_agent_version`: unas pocas versiones vivas a la vez) y es justo lo que permite comparar versiones en el mismo panel. En **Loki** es mejor *structured metadata* que etiqueta de stream: cada release crearía streams nuevos para siempre. La preocupación: una versión nueva por *commit* o por *deploy* haría crecer las series sin límite, así que usa la versión **semántica del prompt/modelo**, no el hash del despliegue.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, json, math, re, time
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import OTelInstrumentor, setup_telemetry
from supportops.quality_eval import (CaseCollector, MockJudge, OnlineEvaluator, RateSampler, StratifiedSampler, VersionTagInstrumentor, confusion, simulate_feedback,
                                     wilson_interval)
from supportops.regression import (PROMPT_VERSIONS, Cusum, GoldenCase, LivePipeline, ReleaseGate, behavior_fingerprint, bootstrap_diff, canary_route, event_records,
                                   golden_to_tickets, js_divergence, ks_2samp, make_agent, mcnemar_exact, min_detectable_drop, novel_topic_tickets,
                                   peeking_false_positive_rate, psi, psi_categorical, replay, required_n_per_arm, run_version, select_golden, sprt_bernoulli,
                                   trace_shapes, trajectory_match, two_proportion_test, version_report)
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded

In [ ]:
N = 800 if FAST_DEV_RUN else 3000
TK = generate_tickets(N, seed=SEED, fault_rates=DEFAULT_FAULT_RATES)         # MISMO tráfico para todas las versiones (comparación apareada)
J = MockJudge(SEED)
RUN = {}
for v in ("v1", "v2", "v2.1"):
    res, cases = run_version(v, TK)
    RUN[v] = {"res": res, "cases": cases, "verd": [J.evaluate(c) for c in cases]}
q = lambda v: float(np.mean([x.scores["groundedness"].label == "pass" for x in RUN[v]["verd"]]))
print({v: f"groundedness OK {q(v):.1%}" for v in RUN})

---
# 1 · Intuición: la regresión silenciosa

La historia es real en forma, no en detalle: un martes se despliega un prompt «más conciso». Se revisan los paneles del módulo 10: latencia ↓, coste ↓, errores = igual. ¡Éxito! Tres semanas
después soporte humano nota que los clientes llaman por números de seguimiento que no existen. **La regresión llevaba tres semanas viva** porque ninguna señal clásica la veía.

| Si vienes de… | Esto es como… | Lo que cambia |
|---|---|---|
| **ML** | *Model drift* / regresión de un modelo reentrenado | El «modelo» es un prompt + un LLM de terceros que puede cambiar **sin avisar**; el conjunto de test hay que **construirlo desde producción** |
| **Backend** | Una migración que cambia el comportamiento sin tocar el contrato | Necesitas tests de **comportamiento** (qué herramientas llama, qué dice) además de tests de contrato |
| **SRE** | Despliegue canario + análisis automático (p. ej. Kayenta) | La métrica de decisión es la **calidad**, que es ruidosa y cara: hay que decidir con pocas muestras |

In [ ]:
fig, ax = viz.canvas(12.5, 4.6, (0, 125), (0, 46))
P = viz.PALETTE
b1 = viz.box(ax, 1, 28, 18, 9, "Producción\n(trazas + eventos\nde evaluación)", P["traces"])
b2 = viz.box(ax, 26, 28, 18, 9, "Dataset de oro\n(fallos, −, escalados,\nmuestra de buenos)", P["logs"], fontsize=8)
b3 = viz.box(ax, 51, 28, 17, 9, "Replay offline\nversión candidata", P["profiles"])
b4 = viz.box(ax, 75, 28, 17, 9, "Compuerta\n(estadística apareada)", P["warn"])
b5 = viz.box(ax, 99, 28, 17, 9, "Canario 5-10 %\n(SPRT)", P["metrics"])
b6 = viz.box(ax, 99, 8, 17, 9, "100 % + vigilancia\n(drift, SLO calidad)", P["ok"])
b7 = viz.box(ax, 51, 8, 41, 9, "Nuevos fallos en producción → vuelven al dataset de oro", P["neutral"], fc="#f3f4f6")
for a, b in ((b1, b2), (b2, b3), (b3, b4), (b4, b5)): viz.arrow(ax, viz.right(a), viz.left(b))
viz.arrow(ax, viz.bottom(b5), viz.top(b6)); viz.arrow(ax, viz.left(b6), viz.right(b7)); viz.arrow(ax, viz.top(b7), viz.bottom(b2), "", P["neutral"], rad=-0.25, dashed=True)
ax.text(1, 43, "El ciclo de regresión: offline barato antes, online prudente después, y producción alimenta el dataset", fontsize=10, fontweight="bold")
ax.text(75, 23.5, "bloquea o pasa", fontsize=8, color=P["warn"]); ax.text(99, 23.5, "promueve o revierte", fontsize=8, color=P["metrics"])
plt.show()

> 👀 **Qué debes observar**
>
> - Hay **dos puertas** antes del 100 %: una barata y *offline* (replay del dataset de oro) y una prudente y *online* (canario con decisión secuencial). Ninguna sustituye a la otra: el replay no ve el tráfico nuevo; el canario es caro y expone a clientes reales.
> - La flecha discontinua es el motor del sistema: **cada fallo en producción alimenta el dataset de oro**. Sin ella, el dataset envejece y deja de representar el tráfico.

## 1.1 Los paneles clásicos mejoran; la calidad empeora

In [ ]:
def fp(v):
    r = RUN[v]["res"]
    return {"latencia media (s)": np.mean([x.latency_s for x in r]), "coste medio (USD)": np.mean([x.cost_usd for x in r]) * 1e3, "% errores": 100 * np.mean([x.status in ("error", "max_steps") for x in r]),
            "herramientas/ticket": np.mean([x.n_tool_calls for x in r]), "groundedness OK (%)": 100 * q(v)}
T = pd.DataFrame({v: fp(v) for v in RUN}).T
display(T.round(3))
fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
for ax, col_ in zip(axes, T.columns):
    ax.bar(T.index, T[col_], color=[viz.PALETTE["ok"], viz.PALETTE["error"], viz.PALETTE["warn"]]); ax.set_title(col_, fontsize=9); ax.tick_params(axis="x", labelsize=8)
    ax.set_ylim(0, T[col_].max() * 1.25)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **v2** tiene mejor latencia, menos coste y **los mismos errores** que v1: un revisor que mire los paneles de los módulos 09/10 la aprobaría. Solo dos columnas delatan el cambio: **menos herramientas por ticket** (el agente ya no consulta) y **menos groundedness**.
> - El «ahorro» de coste y latencia es **el síntoma de la regresión**, no un beneficio: son las llamadas a herramientas que ya no se hacen. Cualquier optimización de coste necesita **guardarraíl de calidad** (conexión con el módulo 13).
> - **v2.1** (el hotfix) recupera casi todo, pero no del todo: más adelante veremos cuántos datos hacen falta para saberlo.

---
# 2 · Teoría: qué cambia, qué se rompe, cómo se detecta

## 2.1 Qué puede regresar (y quién lo cambia)

| Cambio | Quién lo decide | Cómo se ve | Detector principal |
|---|---|---|---|
| Prompt de sistema / plantillas | Tu equipo | Calidad ↓, trayectoria distinta | Replay del dataset + canario |
| Modelo (versión del proveedor) | **El proveedor, a veces sin avisar** (alias `latest`) | Cambio en estilo, tokens, tool use | Canarios sintéticos continuos + huella de comportamiento |
| Herramientas / APIs | Otros equipos | Errores o resultados distintos | Métricas de herramienta (módulo 12), contratos |
| Base de conocimiento / RAG | Contenido | Respuestas desactualizadas, «no encuentro» | Tasa de «sin resultados», groundedness |
| Distribución de entrada (*data drift*) | Los clientes | Temas nuevos, picos estacionales | PSI/KS de la entrada, cobertura del KB |
| Juez / rúbrica | Tu equipo | La «calidad» cambia sin cambiar el agente | Calibración (módulo 14) |

## 2.2 Offline vs online (y por qué hacen falta los dos)

| | Offline (dataset de oro, replay) | Online (canario, A/B, monitorización) |
|---|---|---|
| Tráfico | Fijo, curado | Real, cambiante |
| Coste/riesgo | Barato, sin clientes | Caro, expone clientes |
| Detecta | Regresiones conocidas, cambios deliberados | Efectos de distribución real, fallos nuevos |
| Ciega a | Casos nuevos, interacción con el entorno (herramientas lentas, KB viva) | Lo que no ocurre en la ventana observada |

## 2.3 Tipos de *drift*

| Tipo | Qué cambia | Cómo se mide sin etiquetas |
|---|---|---|
| **Entrada** (covariate shift) | \( P(x) \) | PSI/KS de longitud, tokens, mezcla de intenciones |
| **Concepto** | \( P(y \mid x) \): lo que es «correcto» | Cambios en KB/políticas; requiere etiquetas o juez |
| **Comportamiento** | Qué hace el agente con la misma entrada | Mezcla de herramientas, nº de llamadas, tokens de salida (JS/PSI) |
| **Calidad** | La nota del juez | CUSUM/EWMA sobre la tasa de aprobados |

El **drift de comportamiento** es el más barato de vigilar (no necesita juez) y el que primero delata un cambio de prompt o de modelo.

---
### 📊 Mapa Visual: La Paradoja de la Regresión Silenciosa, Compuerta de Replay y Detección Secuencial CUSUM
![Calidad, Regresión y CUSUM](assets/01_regresion_cusum_y_canarios.svg)

---
# 3 · Dataset de regresión *desde trazas*

La mejor fuente de casos de test es producción. Con la telemetría del módulo 14 sabemos **qué traza falló** (evaluación), **qué traza disgustó** (feedback −) y **cuál escaló**. El algoritmo:

1. Prioriza: `thumbs_down` → `judge_fail` → `error` → `escalated` → `random_pass` (una muestra de los buenos: **sin ellos solo pruebas lo que ya sabes que falla**).
2. **Deduplica** por texto (mismo texto y pedido = mismo caso; se acumula el *peso* = a cuántos tickets representa).
3. **Redacta PII** (`scrub_pii`) y quita el cuerpo con datos personales.
4. Guarda la **trayectoria aceptada** (herramientas usadas en la traza buena) para evaluar el *comportamiento*, no solo el texto.
5. Guarda `source_trace_id`: de cada caso puedes volver a la traza de origen.

In [ ]:
v1 = RUN["v1"]
verdicts = {c.ticket_id: v for c, v in zip(v1["cases"], v1["verd"])}
FB = {f.ticket_id: f.rating for f in simulate_feedback(v1["cases"], seed=SEED)}
tools = {r.ticket_id: r.tools_used for r in v1["res"]}
GOLD = select_golden(v1["cases"], verdicts, FB, n_total=120, tools_by_ticket=tools)
SRC = {t.ticket_id: t for t in TK}; GT = golden_to_tickets(GOLD, SRC)
comp = collections.Counter(g.reason for g in GOLD)
print(f"{len(GOLD)} casos de oro · composición: {dict(comp)} · pesos: representan {sum(g.weight for g in GOLD):.0f} tickets de producción")
display(pd.DataFrame([dataclasses.asdict(g) for g in GOLD[:5]])[["case_id", "reason", "question", "expected_tools", "weight", "source_trace_id"]])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(comp.keys(), comp.values(), color=viz.PALETTE["logs"]); axes[0].set_title("Por qué entró cada caso"); axes[0].tick_params(axis="x", labelsize=8)
iv = collections.Counter(SRC[g.source_ticket_id].intent_true for g in GOLD); ip = collections.Counter(t.intent_true for t in TK)
axes[1].bar(np.arange(4) - 0.2, [iv[k] / len(GOLD) for k in sorted(ip)], 0.4, label="dataset", color=viz.PALETTE["logs"]); axes[1].bar(np.arange(4) + 0.2, [ip[k] / len(TK) for k in sorted(ip)], 0.4, label="producción", color=viz.PALETTE["neutral"])
axes[1].set_xticks(range(4), sorted(ip), fontsize=8); axes[1].legend(); axes[1].set_title("Intenciones: dataset vs producción")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - El dataset está **sesgado a propósito** hacia los fallos (judge_fail, −): es lo que queremos probar. Pero la mezcla de intenciones se desvía de la de producción (más escalados y reembolsos). Por eso cada caso lleva un **peso** y por eso conservamos una muestra de buenos: para poder reponderar a la distribución real y para detectar regresiones en lo que *hoy* funciona.
> - **Privacidad**: el dataset contiene preguntas de clientes. Se redacta PII con `scrub_pii` (módulo 20 amplía a nombres y direcciones) y **no se captura el contenido en las trazas por defecto** (`capture_content=False`): el dataset se construye con una captura *dirigida* (solo los tickets elegidos) y con retención corta.
> - Cada caso guarda `source_trace_id`: si un test falla mañana, abres la traza de producción de la que nació.

> **⚠️ Cuidado · Cuidado: el replay NO reproduce el entorno**
>
> El replay vuelve a ejecutar **la entrada**, no las condiciones de aquel momento: un caso que falló por un 503 del proveedor *no falla* al repetirlo. Por eso (a) los casos de `error` tienen poco valor como test de regresión de *calidad*
> (sirven para pruebas de resiliencia) y (b) los fallos intermitentes se tratan con **repeticiones** o con tolerancia estadística, nunca con un `assert` sobre una sola ejecución.

---
# 4 · Replay y compuerta de release

Repetimos el dataset con cada versión y comparamos. Como las dos versiones ven **los mismos casos**, la comparación es **apareada**: la prueba de McNemar solo mira los casos en los que las versiones discrepan
(\( b \): pasaba con v1 y falla con v2; \( c \): al revés), con p-valor exacto binomial \( 2\cdot P(X \le \min(b,c)),\; X\sim\mathrm{Bin}(b+c,\tfrac12) \).

In [ ]:
REP = {v: replay(GOLD, GT, v, J) for v in ("v1", "v2", "v2.1")}
rows = []
for v, rr in REP.items():
    k = sum(r["passed"] for r in rr); lo, hi = wilson_interval(k, len(rr))
    rows.append({"versión": v, "calidad OK": k / len(rr), "IC95 bajo": lo, "IC95 alto": hi, "trayectoria OK": np.mean([r["trajectory_ok"] for r in rr]), "coste medio": np.mean([r["cost_usd"] for r in rr]), "latencia": np.mean([r["latency_s"] for r in rr])})
R = pd.DataFrame(rows).set_index("versión"); display(R.round(4))
gate = ReleaseGate()
for cand in ("v2", "v2.1"):
    dec, why = gate.decide(REP["v1"], REP[cand])
    b = sum(a["passed"] and not c["passed"] for a, c in zip(REP["v1"], REP[cand])); c_ = sum((not a["passed"]) and c["passed"] for a, c in zip(REP["v1"], REP[cand]))
    print(f"v1 → {cand}: {dec.upper():5s} · McNemar b={b} c={c_} p={mcnemar_exact(b, c_):.4f} · {why or 'sin motivos'}")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
x = np.arange(len(R)); ax.errorbar(x, R["calidad OK"], yerr=[R["calidad OK"] - R["IC95 bajo"], R["IC95 alto"] - R["calidad OK"]], fmt="o", capsize=6, color=viz.PALETTE["traces"], label="calidad OK (IC95 Wilson)")
ax.plot(x, R["trayectoria OK"], "s--", color=viz.PALETTE["profiles"], label="trayectoria correcta"); ax.set_xticks(x, R.index); ax.set_ylim(0.6, 1.02); ax.legend(); ax.set_title("Replay del dataset de oro por versión")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La compuerta **bloquea v2** por dos motivos independientes: la calidad cae ≈ 17 puntos (McNemar: ≈ 20 casos empeoran, ninguno mejora, p ≈ 1e-6) y la **trayectoria** (qué herramientas llama) se desvía de la aceptada. La segunda señal no necesita juez.
> - **v2.1 (el hotfix) queda en el filo**: pierde ≈ 4-5 puntos, 5-6 casos empeoran y ninguno mejora (p ≈ 0,03-0,06). Según cuál sea la muestra de 120 casos la compuerta lo bloquea o lo deja pasar: **esa fragilidad es lo que significa «poco poder estadístico»** (sección 5). Los intervalos de Wilson de v1 y v2.1 se solapan: una comparación *no apareada* ni se acercaría a detectarlo.
> - Aparear —que cada caso sea su propio control— multiplica el poder. Por eso `ReleaseGate` usa McNemar cuando las dos ejecuciones son de los mismos casos y el IC de la diferencia de proporciones solo cuando no lo son.

---
# 5 · Poder estadístico: ¿cuántos datos hacen falta?

Para decidir «v2.1 no es peor que v1» hay que fijar de antemano qué diferencia te importa. Con dos proporciones \(p_0\) (base) y \(p_1\) (candidata), nivel \(\alpha\) y poder \(1-\beta\):
$$ n \;=\; \frac{\Big(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)} + z_{1-\beta}\sqrt{p_0(1-p_0)+p_1(1-p_1)}\Big)^2}{(p_0-p_1)^2}, \qquad \bar p = \tfrac{p_0+p_1}{2}$$
por grupo. Dicho al revés: con \(n\) por grupo, la **caída mínima detectable** es \(\approx (z_{1-\alpha/2}+z_{1-\beta})\sqrt{2p_0(1-p_0)/n}\).

In [ ]:
drops = np.array([0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8))
for p0, col_ in ((0.97, viz.PALETTE["traces"]), (0.90, viz.PALETTE["metrics"]), (0.80, viz.PALETTE["profiles"])):
    axes[0].plot(drops * 100, [required_n_per_arm(p0, p0 - d) for d in drops], "o-", label=f"calidad base {p0:.0%}", color=col_)
axes[0].set_yscale("log"); axes[0].set_xlabel("caída de calidad a detectar (puntos)"); axes[0].set_ylabel("n por grupo (α=5 %, potencia 80 %)"); axes[0].legend(); axes[0].set_title("Casos necesarios")
ns = np.array([30, 60, 120, 250, 500, 1000, 3000])
axes[1].plot(ns, [min_detectable_drop(int(n), 0.97) * 100 for n in ns], "o-", color=viz.PALETTE["traces"], label="base 97 %")
axes[1].plot(ns, [min_detectable_drop(int(n), 0.85) * 100 for n in ns], "o-", color=viz.PALETTE["metrics"], label="base 85 %")
axes[1].axvline(120, color=viz.PALETTE["neutral"], ls=":"); axes[1].text(125, axes[1].get_ylim()[1] * 0.85, "nuestro dataset (120)", fontsize=8)
axes[1].set_xscale("log"); axes[1].set_xlabel("n por grupo"); axes[1].set_ylabel("caída mínima detectable (puntos)"); axes[1].legend(); axes[1].set_title("Sensibilidad según el tamaño del dataset")
plt.tight_layout(); plt.show()
print("n por grupo para detectar 97% → 92%:", required_n_per_arm(0.97, 0.92), "· 90% → 85%:", required_n_per_arm(0.90, 0.85), "· caída mínima con 120 casos y base 97%:", f"{min_detectable_drop(120, 0.97):.1%}")

> 👀 **Qué debes observar**
>
> - Detectar una caída pequeña es **caro**: 5 puntos desde 90 % requieren unos 700 casos *por grupo*; desde 97 %, menos (la varianza es menor cerca del techo) pero aun así cientos. Con 120 casos *independientes* solo ves caídas de ≥ 6 puntos si la base es ≈ 97 % (≈ 13 si es ≈ 85 %): v2 (−16) sí, v2.1 (−5) no… salvo que compares de forma **apareada**, como en la sección 4, donde v2.1 queda al límite de la significación (p ≈ 0,03-0,06).
> - Dos formas de ganar poder sin más datos: **comparación apareada** (cada caso es su propio control) y **métricas continuas/más finas** (puntuaciones del juez, no pass/fail).
> - Conclusión práctica: el dataset de oro es una **red de seguridad para regresiones grandes**; las pequeñas se vigilan online con volumen (canario + SLO de calidad, módulo 16).

## 5.1 El problema de «mirar antes de tiempo» (*peeking*)

In [ ]:
pk = peeking_false_positive_rate(p=0.06, n_max=600, looks=20, trials=300, seed=SEED)
print(pk)
fig, ax = plt.subplots(figsize=(5.8, 3.4))
ax.bar(["nominal", "prueba z repetida\n20 veces", "SPRT"], [pk["nominal"], pk["fixed_sample_peeking"], pk["sprt"]], color=[viz.PALETTE["neutral"], viz.PALETTE["error"], viz.PALETTE["ok"]])
ax.set_ylabel("falsos positivos (dos versiones IDÉNTICAS)"); ax.set_title("Mirar sin corregir infla el error")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Con dos grupos **idénticos**, repetir una prueba de tamaño fijo cada 30 peticiones y parar en el primer «significativo» declara regresión en ≈ 20–25 % de los casos en lugar del 5 % prometido: cada mirada es una nueva oportunidad de error (Johari et al., 2017).
> - El **SPRT** de Wald está diseñado para mirar tras cada observación y mantiene el error cerca del nominal. Es lo que usamos en el canario (sección 7).

---
# 6 · Drift: de entrada, de comportamiento y de calidad

## 6.1 Drift de entrada: temas nuevos
Inyectamos tickets sobre temas que la base de conocimiento no cubre (reciclaje, facturación en otro país…). Sin etiquetas, ¿qué se ve?

In [ ]:
NOV = novel_topic_tickets(len(TK) // 4, seed=7)
res_n, cases_n = run_version("v1", NOV)
mix = lambda r: collections.Counter(t for x in r for t in x.tools_used)
len_ref = [len(t.text) for t in TK]; len_nov = [len(t.text) for t in NOV]
rows = [
    {"magnitud": "longitud de la pregunta (PSI)", "valor": psi(len_ref, len_nov), "umbral": 0.25},
    {"magnitud": "longitud de la pregunta (KS p-valor)", "valor": ks_2samp(len_ref, len_nov)[1], "umbral": 0.01},
    {"magnitud": "mezcla de herramientas (JS)", "valor": js_divergence(mix(RUN["v1"]["res"]), mix(res_n)), "umbral": 0.10},
    {"magnitud": "tasa de «no encuentro» en las respuestas", "valor": np.mean([("No he encontrado" in x.answer) for x in res_n]), "umbral": 0.10},
    {"magnitud": "groundedness OK (juez)", "valor": np.mean([J.evaluate(c).scores["groundedness"].label == "pass" for c in cases_n]), "umbral": 0.90},
    {"magnitud": "calidad global OK (juez)", "valor": np.mean([J.evaluate(c).passed for c in cases_n]), "umbral": 0.80},
]
display(pd.DataFrame(rows).set_index("magnitud").round(4))

> 👀 **Qué debes observar**
>
> - La entrada derivó (PSI de longitud muy por encima de 0,25) y la **mezcla de herramientas** cambió (casi todo `search_kb`): ambas señales aparecen **sin juez**.
> - Un hallazgo contraintuitivo: la **groundedness** sigue alta (el agente no inventa: dice «no he encontrado información») mientras la **calidad global** cae por relevancia/resolución. Un agente «honesto pero inútil» no es una alucinación: necesitas varios criterios, no uno.
> - Qué hacer con este drift no es revertir una versión, sino **ampliar la base de conocimiento** o la cobertura: la señal es para el producto, no para el despliegue.

## 6.2 Drift de comportamiento: la huella de una versión

In [ ]:
fps = pd.DataFrame({v: behavior_fingerprint([dict(r.to_dict(), n_tool_calls=r.n_tool_calls) for r in RUN[v]["res"]]) for v in RUN}).T.drop(columns="n")
display(fps.round(4))
base_mix = mix(RUN["v1"]["res"])
js = {v: js_divergence(base_mix, mix(RUN[v]["res"])) for v in ("v2", "v2.1")}
psi_tok = {v: psi([x.output_tokens for x in RUN["v1"]["res"]], [x.output_tokens for x in RUN[v]["res"]]) for v in ("v2", "v2.1")}
print("JS de la mezcla de herramientas frente a v1:", {k: round(x, 4) for k, x in js.items()}, "· PSI de tokens de salida:", {k: round(x, 3) for k, x in psi_tok.items()})
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.5))
for v, col_ in (("v1", viz.PALETTE["ok"]), ("v2", viz.PALETTE["error"]), ("v2.1", viz.PALETTE["warn"])):
    nt = collections.Counter(x.n_tool_calls for x in RUN[v]["res"]); ks = sorted(nt)
    axes[0].plot(ks, [nt[k] / N for k in ks], "o-", label=v, color=col_)
axes[0].set_xlabel("llamadas a herramientas por ticket"); axes[0].set_ylabel("fracción de tickets"); axes[0].legend(); axes[0].set_title("Distribución del nº de herramientas")
axes[1].bar(["no_tool_rate v1", "v2", "v2.1"], [fps.loc[v, "no_tool_rate"] for v in ("v1", "v2", "v2.1")], color=[viz.PALETTE["ok"], viz.PALETTE["error"], viz.PALETTE["warn"]]); axes[1].set_title("Tickets sin ninguna herramienta")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La **huella de comportamiento** separa a v2 de v1 sin mirar ni una respuesta: el porcentaje de tickets *sin ninguna herramienta* se triplica y la distribución del nº de llamadas se desplaza hacia 0.
> - v2.1 casi restaura la huella (`no_tool_rate` ≈ 11,5 % frente a ≈ 8,4 % de v1) pero **no del todo**: sigue habiendo tickets sin herramienta que antes no los había. Es la ventaja del monitor de comportamiento: usa **todo el tráfico**, no un dataset pequeño.
> - La divergencia JS de la mezcla *global* de herramientas es ≈ 0 en las dos versiones (la mezcla casi no cambia: lo que cambia es **cuántos tickets no llaman a nada**); el PSI de los tokens de salida de v2 (≈ 0,24, cerca del umbral habitual de 0,25) sí lo capta. **Elige la magnitud que corresponde al fallo**, no una distancia genérica: una divergencia sobre la mezcla de herramientas no ve un agente que deja de usarlas todas por igual.

## 6.3 Drift de calidad en el tiempo: CUSUM

In [ ]:
# Serie temporal: despliegue de v2 en el minuto 40; cada punto = pass-rate de groundedness del minuto sobre ≈ 10 evaluaciones (calidad 93 % → 82 %)
def serie(seed, minutes=100, onset=40, n=10, p0=0.93, p1=0.82):
    rng = np.random.default_rng(seed); return np.array([rng.binomial(n, p0 if m < onset else p1) / n for m in range(minutes)])
ser = serie(SEED)
K, H = 0.05, 0.6
cus = Cusum(target=0.93, k=K, h=H); alarm = cus.first_alarm(ser)
S, s_ = [], 0.0
for x in ser: s_ = max(0.0, s_ + (0.93 - x) - K); S.append(s_)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(ser, color=viz.PALETTE["traces"], lw=1); axes[0].axvline(40, color=viz.PALETTE["error"], ls="--"); axes[0].set_ylabel("pass-rate por minuto"); axes[0].set_title("Calidad muestreada por minuto (n≈10 por punto)")
axes[1].plot(S, color=viz.PALETTE["metrics"]); axes[1].axhline(H, color=viz.PALETTE["error"], ls=":"); axes[1].set_ylabel("CUSUM"); axes[1].set_xlabel("minuto")
if alarm is not None: axes[1].axvline(alarm, color=viz.PALETTE["ok"], ls="--"); axes[1].text(alarm + 1, H * 1.1, f"alarma min {alarm}", fontsize=9)
plt.tight_layout(); plt.show()
# Monte Carlo (200 repeticiones): falsas alarmas ANTES del despliegue y retraso de detección DESPUÉS
def mc(detector):
    fa, dl = 0, []
    for sd in range(200):
        t = detector(serie(sd))
        if t is not None and t < 40: fa += 1
        elif t is not None: dl.append(t - 40)
    return {"falsas alarmas": fa / 200, "retraso mediano (min)": float(np.median(dl)) if dl else None, "no detectado": 1 - (fa + len(dl)) / 200}
ma = lambda thr: (lambda x: next((i for i in range(4, len(x)) if x[i - 4:i + 1].mean() < thr), None))
table = pd.DataFrame({"media móvil 5 min < 0,85": mc(ma(0.85)), "media móvil 5 min < 0,80": mc(ma(0.80)), f"CUSUM (k={K}, h={H})": mc(lambda x: Cusum(0.93, K, H).first_alarm(x))}).T
display(table.round(3))

> 👀 **Qué debes observar**
>
> - Cada punto de la serie es muy ruidoso (n≈10 evaluaciones por minuto). La tabla Monte Carlo muestra la disyuntiva de siempre: un umbral sensible (< 0,85) detecta en ≈ 3 min pero **da falsa alarma en más de un tercio de las series sanas**; endurecerlo (< 0,80) baja las falsas alarmas al ≈ 5 % a costa de tardar ≈ 8 min.
> - **CUSUM acumula la evidencia** (\( S_t=\max(0,S_{t-1}+(\mu_0-x_t)-k) \)): con un retraso mediano parecido (≈ 7-8 min) tiene una tasa de falsas alarmas menor (≈ 2-3 %). No es magia: es mejor uso de los mismos datos.
> - Los parámetros \(k\) (≈ la mitad de la caída que quieres detectar) y \(h\) (cuánta evidencia acumulada) se calibran con la serie histórica *sana*. Es la misma disyuntiva que la alerta por *burn rate* del módulo 16.

---
# 7 · Canarios: decidir con pocos datos

Un **canario** manda una fracción pequeña (5–10 %) del tráfico a la versión nueva, **enrutada de forma determinista** por hash del ticket (mismo cliente, misma versión: sin parpadeos y sin mezclar conversaciones).
La decisión se toma con una prueba **secuencial**: tras cada petición se actualiza el log-cociente de verosimilitud del contraste *tasa sana \( p_0 \)* frente a *tasa mala \( p_1 \)*:
$$ \Lambda_n = k\,\ln\frac{p_1}{p_0} + (n-k)\,\ln\frac{1-p_1}{1-p_0}, \qquad \text{revertir si } \Lambda_n \ge \ln\frac{1-\beta}{\alpha},\quad \text{promover si } \Lambda_n \le \ln\frac{\beta}{1-\alpha}$$
con \( k \) fallos de calidad en \( n \) peticiones evaluadas.

In [ ]:
share = np.mean([canary_route(t.ticket_id, 0.10) == "canary" for t in TK]); print(f"fracción enrutada al canario: {share:.1%} (objetivo 10 %) · determinista:", canary_route("T-4200001") == canary_route("T-4200001"))
P0, P1 = 0.07, 0.15                    # tasa sana de fallos de groundedness (≈ 7 %) vs la que consideramos regresión (15 %)
traj = {}
for v in ("v1", "v2", "v2.1"):
    fails = [x.scores["groundedness"].label != "pass" for x in RUN[v]["verd"]]
    k, path, dec = 0, [], None
    for i, f in enumerate(fails, 1):
        k += f; d, llr = sprt_bernoulli(k, i, P0, P1, alpha=0.05, beta=0.2); path.append(llr)
        if d != "continue" and dec is None: dec = (d, i)
    traj[v] = (path, dec)
up, lo = math.log(0.8 / 0.05), math.log(0.2 / 0.95)
fig, ax = plt.subplots(figsize=(9, 4))
for v, col_ in (("v1", viz.PALETTE["ok"]), ("v2", viz.PALETTE["error"]), ("v2.1", viz.PALETTE["warn"])):
    ax.plot(range(1, 301), traj[v][0][:300], label=f"{v}: {traj[v][1]}", color=col_)
ax.axhline(up, color=viz.PALETTE["error"], ls=":"); ax.axhline(lo, color=viz.PALETTE["ok"], ls=":"); ax.text(200, up + 0.15, "revertir", fontsize=9); ax.text(200, lo - 0.6, "promover", fontsize=9)
ax.set_xlabel("evaluaciones del canario (n)"); ax.set_ylabel("log-cociente de verosimilitud Λ"); ax.set_ylim(lo - 2, up + 2); ax.legend(); ax.set_title("SPRT: cuántos datos hacen falta para decidir")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **v2 se revierte con muy pocas evaluaciones** (decenas): su tasa de fallo es 3-4× la sana y el SPRT lo ve rápido. **v1** (sana) se promueve con unos cientos. La asimetría es la gracia: lo claramente malo se corta pronto, lo bueno requiere más pruebas.
> - **v2.1** (≈ 10 % de fallos frente al 7 % sano) es la zona gris: depende de qué \(p_1\) hayas declarado «inaceptable». Con \(p_1=15\\%\) tarda más en decidir o se promueve; con \(p_1=10\\%\) se revertiría. **Declarar \(p_0\) y \(p_1\) de antemano es una decisión de producto**, no de estadística.
> - Para el 5-10 % de tráfico a una cadencia de 1 000 tickets/hora y evaluando el 25 %, 100 evaluaciones llegan en unas 4 horas: el canario cuesta **tiempo**; por eso se acompaña de la compuerta offline.

---
# 8 · Detectar la regresión usando solo telemetría

Situación realista: no tienes acceso al código ni al prompt; solo a **spans y eventos de evaluación** en Tempo/Loki. Se han desplegado dos versiones de forma consecutiva (tráfico distinto). ¿Cuál regresó y qué trazas lo prueban?
Generamos la telemetría con el `OTelInstrumentor` (etiquetas de versión `gen_ai.agent.version`) y el evaluador del módulo 14 (muestra del 30 %).

In [ ]:
def run_telemetry(version, tickets, rate=0.3):
    ta = setup_telemetry("supportops-agent", mode="memory", set_global=False)
    te = setup_telemetry("supportops-evaluator", mode="memory", set_global=False)
    ev = OnlineEvaluator(J, RateSampler(rate, "tel"), te)
    agent, col = make_agent(version, instrumentors=[OTelInstrumentor(ta), VersionTagInstrumentor(version)], collector=CaseCollector(version, on_case=ev.submit))
    res = [agent.run(t) for t in tickets]; ev.run_pending(); ta.force_flush(); te.force_flush()
    return ta.span_records(), event_records(te), res, col.cases

TA, TB = generate_tickets(500, seed=11, fault_rates=DEFAULT_FAULT_RATES), generate_tickets(500, seed=12, fault_rates=DEFAULT_FAULT_RATES)
sp1, ev1, r1, c1 = run_telemetry("v1", TA); sp2, ev2, r2, c2 = run_telemetry("v2", TB)
SPANS, EVENTS = sp1 + sp2, ev1 + ev2
SHAPES = trace_shapes(SPANS)
REP_V = pd.DataFrame(version_report(SHAPES, EVENTS)).set_index("version")
display(REP_V[["n", "p50_s", "p95_s", "error_rate", "tools_per_ticket", "no_tool_rate", "tokens_per_ticket", "groundedness_pass", "groundedness_n", "relevance_pass", "resolution_pass"]].round(3))

> 👀 **Qué debes observar**
>
> - Mirando **solo lo clásico** (p50/p95, error_rate, tokens) v2 parece igual o mejor. Las dos columnas que lo delatan son `tools_per_ticket`/`no_tool_rate` (estructura de la traza, **sin juez**) y `groundedness_pass` (evento de evaluación).
> - La calidad se mide solo en la **muestra** evaluada (`groundedness_n` ≈ 150 por versión): es suficiente para ver una caída de ≥ 10 puntos, no una de 2.
> - Todo esto sale de `gen_ai.agent.version` en los spans y en los eventos: **etiquetar la versión en toda la telemetría** es lo que hace posible comparar. Sin la etiqueta habría que inferir el despliegue por la fecha.

In [ ]:
t = two_proportion_test(int(round(REP_V.loc["v1", "groundedness_pass"] * REP_V.loc["v1", "groundedness_n"])), int(REP_V.loc["v1", "groundedness_n"]),
                        int(round(REP_V.loc["v2", "groundedness_pass"] * REP_V.loc["v2", "groundedness_n"])), int(REP_V.loc["v2", "groundedness_n"]))
print(f"groundedness v1 {t['p1']:.1%} → v2 {t['p2']:.1%} · diferencia {t['diff']:+.1%} IC95 [{t['ci_low']:+.1%}, {t['ci_high']:+.1%}] · p = {t['p_value']:.2g}")
# ¿Qué trazas lo explican? Tickets de intención que necesita herramienta, sin ninguna llamada a herramienta, en v2
suspects = [s for s in SHAPES.values() if s["version"] == "v2" and s["n_tool"] == 0 and s["status"] == "ok" and s["intent"] in ("order_status", "refund", "kb_question")]
suspects_v1 = [s for s in SHAPES.values() if s["version"] == "v1" and s["n_tool"] == 0 and s["status"] == "ok" and s["intent"] in ("order_status", "refund", "kb_question")]
truth_reg = {c.ticket_id for c in c2 if "prompt_regression" in c.truth["faults"]}
prec = np.mean([s["ticket"] in truth_reg for s in suspects]); rec = np.mean([any(s["ticket"] == t_ for s in suspects) for t_ in truth_reg])
print(f"trazas sospechosas (v2, status ok, 0 herramientas, intención que las necesita): {len(suspects)} (en v1 con el mismo filtro: {len(suspects_v1)}) · precisión frente a la verdad {prec:.0%} · exhaustividad {rec:.0%}")
sample_trace = suspects[0]["trace_id"]; ok_trace = next(s["trace_id"] for s in SHAPES.values() if s["version"] == "v1" and s["n_tool"] >= 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 2.7))
for ax, tid, ttl in ((axes[0], ok_trace, "v1: traza normal (consulta herramienta)"), (axes[1], sample_trace, "v2: traza sospechosa (contesta sin consultar)")):
    viz.waterfall([s for s in SPANS if s["trace_id"] == tid], ax=ax, title=ttl)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La comparación de calidad es significativa (el IC de la diferencia no contiene 0). Las trazas con **cero herramientas en intenciones que las necesitan** localizan la regresión: ≈ 22 % de los tickets de v2 frente a ≈ 7 % de v1 con el mismo filtro (la diferencia, ≈ 15 puntos, coincide con la caída de groundedness). La causa raíz queda a la vista **sin leer el prompt**: la cascada de v2 se queda en un único span `chat`.
> - El detector estructural es un **proxy**, no una prueba: su precisión frente a la verdad ronda el 70 % (el resto son peticiones legítimas sin herramienta, como pedir al cliente su nº de pedido, y alucinaciones de fondo que también existen en v1) y no cubre regresiones que no cambian la trayectoria. Se usa para *localizar* y *priorizar*; la calidad la confirma el evento de evaluación.

### Las consultas de Tempo y Loki que usarías

```text
# Tempo (TraceQL): tickets de v2 que debían usar una herramienta y no la usaron
{ name =~ "invoke_agent.*" && span.gen_ai.agent.version = "v2" && span.supportops.tool_calls = 0 && span.supportops.intent =~ "order_status|refund" }

# Loki (LogQL): evaluaciones fallidas de groundedness por versión
sum by (gen_ai_agent_version) (count_over_time({service_name="supportops-evaluator"} | gen_ai_evaluation_name="groundedness" | gen_ai_evaluation_score_label="fail" [5m]))

# Prometheus (PromQL): calidad por versión
sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[5m]))
  / sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[5m]))
```

---
# 9 · Con el stack real: un despliegue en canario, en vivo

`LivePipeline` envía ≈ 60 s de tráfico al Collector con el flujo: los primeros ≈ 20 s todo es v1; después **v2 recibe el 40 % del tráfico (canario)**. Veremos la regresión en tres backends distintos.

In [ ]:
LIVE_S = 60 if FAST_DEV_RUN else 150
if LIVE:
    from supportops.slo import BASELINE_FAULTS
    lp = LivePipeline(("v1", "v2"), mode="otlp", sampler=RateSampler(0.6), metric_interval_s=5)
    tk_live = generate_tickets(int(LIVE_S * 4.5), seed=SEED, duration_s=LIVE_S, fault_rates=BASELINE_FAULTS, start_s=0.0)
    t0 = time.time(); lp.run_live(tk_live, LIVE_S, lambda t, el: "v2" if el > LIVE_S / 3 and canary_route(t.ticket_id, 0.40) == "canary" else "v1"); lp.close()
    print(f"enviados {len(tk_live)} tickets en {time.time() - t0:.0f}s · tráfico por versión: {collections.Counter(c.version for c in lp.cases)}")
    time.sleep(14)
else:
    print("Modo degradado: levanta el stack con:", STACK.command)

In [ ]:
if LIVE:
    from supportops.otel_metrics import prom_instant
    qv = 'sum by (gen_ai_agent_version) (increase(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[10m])) / sum by (gen_ai_agent_version) (increase(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[10m]))'
    print("Prometheus · calidad por versión:", {r["metric"].get("gen_ai_agent_version"): round(r["value"], 3) for r in prom_instant(qv)})
    q_tools = 'sum by (gen_ai_agent_version) (increase(supportops_eval_population_total[10m]))'
    print("Prometheus · peticiones por versión:", {r["metric"].get("gen_ai_agent_version"): round(r["value"]) for r in prom_instant(q_tools)})
    tq = '{ name =~ "invoke_agent.*" && span.gen_ai.agent.version = "v2" && span.supportops.tool_calls = 0 && span.supportops.intent =~ "order_status|refund" }'
    ts = backends.tempo_search(tq, minutes=15, limit=20)
    print("Tempo · trazas v2 sin herramientas en pedidos/reembolsos:", len(ts.get("traces", [])))
    lg = backends.loki_query('{service_name="supportops-evaluator"} | gen_ai_evaluation_name="groundedness" | gen_ai_evaluation_score_label="fail" | gen_ai_agent_version="v2"', minutes=15, limit=50)
    print("Loki · evaluaciones fallidas de groundedness en v2:", len(lg))

### El dashboard `15-regresion-versiones`

`stack/grafana/dashboards/15-regresion-versiones.json` (uid `regresion-versiones`) junta las tres lecturas: **calidad por versión** (PromQL sobre los eventos agregados), **comportamiento** (herramientas por ticket, latencia y coste *clásicos*, sin etiqueta de versión pero con el escalón del despliegue) y las **trazas sospechosas** (TraceQL).
**Qué debes ver:** al entrar v2, la línea de v2 en «Calidad por versión» cae por debajo de v1 mientras los paneles de latencia y coste muestran una ligera **mejora**; la tabla de Tempo se llena de trazas de un solo span `chat`.

---
# 10 · 🏭 En producción

- **Evals en CI/CD.** El dataset de oro se ejecuta en cada cambio de prompt, modelo o herramienta como una suite de pruebas (la «compuerta»), con resultados guardados por versión. Las plataformas de evaluación comerciales y abiertas (Langfuse, LangSmith, Braintrust, Arize Phoenix, promptfoo, OpenAI Evals…) implementan variantes de este ciclo: datasets versionados, ejecuciones comparables y gating; lo que aporta OTel es que la **misma telemetría** sirve para observar y para construir los datasets.
- **Versionado de prompts.** El prompt es código: versión semántica o hash, registro de cambios y etiquetado en la telemetría (`gen_ai.prompt.name`/`gen_ai.prompt.version`, spec GenAI *Development*; `gen_ai.agent.version`). Sin etiqueta no hay comparativa.
- **Fijar el modelo.** Evita alias flotantes (`latest`) en producción: el proveedor puede cambiar el modelo por debajo. Si no puedes fijarlo, ejecuta **canarios sintéticos** cada pocos minutos (un conjunto fijo de prompts con aserciones) y alerta si cambia la huella.
- **Canarios y progresivos.** Despliegue progresivo (5 → 25 → 50 → 100 %) con análisis automático en cada escalón; el análisis compara canario y línea base *al mismo tiempo* (no contra «ayer») para neutralizar la estacionalidad.
- **Coste de las evals.** Las regresiones se ejecutan sobre dataset pequeño con juez pequeño/reglas y se escalan solo si la señal es ambigua (módulos 13, 14 y 18).

---
# 11 · 🧠 Secretos de la élite

1. **Las mejoras de latencia y coste tras un cambio de prompt son sospechosas hasta que se demuestre lo contrario**: suelen ser llamadas a herramientas o razonamiento que dejó de ocurrir. Todo cambio «de eficiencia» necesita guardarraíl de calidad y de trayectoria.
2. **Evalúa la trayectoria, no solo la respuesta.** Comparar el conjunto/orden de herramientas llamadas con el de la traza aceptada detecta regresiones sin juez, baratas y deterministas (`trajectory_match`). El texto puede cambiar sin ser peor; la trayectoria cambia cuando cambia el comportamiento.
3. **Compara apareado y al mismo tiempo.** Mismo conjunto de casos (replay) o mismo periodo (canario simultáneo). Comparar «esta semana» con «la semana pasada» mezcla la regresión con la estacionalidad y los cambios de tráfico.
4. **Fija el tamaño de muestra y la diferencia relevante antes de mirar**, o usa una prueba secuencial. Mirar repetidamente y parar al primer p < 0,05 infla el error de tipo I muy por encima del 5 % (en nuestra simulación de 20 miradas, ≈ 1 de cada 4; el efecto crece con el nº de miradas, ver Johari et al., 2017).
5. **Un dataset de oro envejece**: pesa los casos por su frecuencia real, retira los obsoletos, añade cada incidente nuevo y vigila la distancia entre su distribución y la de producción (PSI). Un dataset que ya no se parece a producción da luz verde a regresiones reales.
6. **Los casos de `error` no son tests de calidad.** Un 503 del proveedor no se reproduce en un replay; mezclarlos infla la varianza. Sepáralos en una suite de resiliencia.
7. **Etiqueta la versión en TODAS las señales** (spans, logs, métricas, eventos de evaluación) y mantén la cardinalidad acotada (versión ≈ decenas de valores, no un hash por despliegue de CI). Sin esa etiqueta, la regresión se descubre por correlación temporal con el calendario de despliegues.
8. **Rollback automático solo con datos suficientes.** Un SPRT con \(p_0,p_1\) declarados evita revertir por ruido y evita esperar 3 semanas; pero necesita una definición de «fallo» estable (juez versionado, módulo 14).
9. **Cuenta el coste del hotfix.** v2.1 recuperó casi todo… pero «casi» sobre millones de tickets son miles de respuestas malas: calcula el impacto (tasa residual × volumen) antes de dar un incidente por cerrado.
10. **Haz primero un A/A: la versión idéntica contra sí misma.** Antes de creer una diferencia v1-v2, enruta el tráfico a dos réplicas de la **misma** versión y comprueba que el reparto real coincide con el planificado (*sample ratio mismatch*, SRM) y que la métrica no «detecta» diferencias. Con LLM la salida varía entre ejecuciones incluso con el mismo prompt (Anthropic: los agentes son «non-deterministic between runs, even with identical prompts»), así que la varianza intra-versión es parte de tu ruido. Un A/A que dispara falsos positivos > 5 % invalida el juez, el reparto o el etiquetado. *Fuente: Kohavi, Tang, Xu (2020), capítulos sobre SRM y pruebas A/A; Anthropic Engineering (2025) para el no determinismo.*
11. **Si vas a mirar siempre, usa inferencia válida en cualquier momento, no un p-valor cada hora.** Johari et al. no solo documentan el problema: proponen *p-valores e intervalos siempre válidos* basados en una prueba secuencial de razón de probabilidades por mezcla (mSPRT), que mantienen el error de tipo I bajo control con mirada continua a cambio de algo de potencia; versión extendida en *Operations Research* (2022). El SPRT de Wald de este módulo es el caso de hipótesis simples. *Fuente: Johari, Koomen, Pekelis, Walsh (KDD '17, pp. 1517-1525; confirmado en DBLP); la versión de* Operations Research *se confirmó solo por el listado de DBLP.*
12. **Un canario necesita un grupo de control concurrente y tres decisiones que se compensan.** El Workbook define el canario como un despliegue parcial y limitado en el tiempo, evaluado contra el resto (el «control»). La charla de SREcon18 de Štěpán Davidovič (Google SRE) lo resume en un triángulo: **duración** (más datos, menos velocidad), **tamaño de la población** (más representativo, pero más daño si es malo) y **selección de métricas** (más métricas, más ruido y falsas alarmas). Aquí: reparto del canario (módulo 15 §9), ventana de decisión y qué señales cuentan (calidad con intervalo, trayectoria, coste por ticket). *Fuente: SRE Workbook cap. 16 «Canarying Releases» (definición leída en el extracto de O'Reilly); el triángulo, vía el resumen de New Relic de la charla (fuente secundaria, no verificada en el original).*

---
# 12 · ⚠️ Errores comunes

- Probar solo con el dataset de **fallos pasados** (sin una muestra de lo que funciona): no ves regresiones en lo que hoy va bien.
- Mezclar fallos de infraestructura con fallos de calidad en el mismo dataset.
- Comparar versiones con **tráfico distinto** (otro día, otro mix) y atribuir la diferencia al prompt.
- Declarar «no hay regresión» porque *p* > 0,05 con 50 casos (ausencia de evidencia ≠ evidencia de ausencia): calcula la caída mínima detectable.
- Cambiar el juez a la vez que el agente (no sabrás cuál causó el cambio).
- Guardar contenido de clientes en el dataset sin redactar ni caducidad.
- Canario sin enrutamiento determinista: el mismo cliente ve una versión u otra a mitad de una conversación.

---
# 13 · Autoevaluación

<details>
<summary>1 · (Cálculo) Quieres detectar una caída de calidad de 0,90 a 0,85 con α = 5 % y potencia 80 %. ¿Cuántos casos por versión? ¿Cuántos si la base fuera 0,97 y la caída 0,92?</summary>

Con la fórmula de dos proporciones: ≈ 686 por grupo (90 → 85) y ≈ 326 por grupo (97 → 92) con esta aproximación. La varianza \( p(1-p) \) es menor cerca de 1, así que la misma caída de 5 puntos requiere menos casos cerca del techo. Con un dataset de 120 casos solo detectas caídas de ≥ 6 puntos (base 97 %) o ≈ 13 (base 85 %).

</details>

<details>
<summary>2 · (Cálculo) Tras el replay, 14 casos pasaban con v1 y fallan con v2, y 1 caso al revés. ¿Es significativa la regresión?</summary>

McNemar exacto: b = 14, c = 1 → n = 15, min = 1. p = 2·P(Bin(15, ½) ≤ 1) = 2·(1 + 15)/32768 ≈ 0,001. Sí: significativa. Solo cuentan los casos discordantes; los que pasan o fallan en ambas versiones no aportan información sobre la diferencia.

</details>

<details>
<summary>3 · (Diagnóstico) Tras un despliegue el coste por ticket baja un 12 %, el p95 baja un 8 % y los errores no cambian. El equipo celebra. ¿Qué miras antes de aceptar?</summary>

Calidad (groundedness/relevancia por versión) y trayectoria: herramientas por ticket, `no_tool_rate`, tokens de salida. Un ahorro sin cambio de modelo suele ser trabajo que se dejó de hacer. Comprueba también la mezcla de intenciones (¿cambió el tráfico?) para descartar un efecto de composición.

</details>

<details>
<summary>4 · (Diagnóstico) El replay de tu dataset de oro pasa al 100 % con la versión nueva, pero en producción los clientes se quejan. Da tres causas.</summary>

(1) El dataset no representa el tráfico actual (drift de entrada, falta de casos nuevos). (2) Los fallos dependen del entorno que el replay no reproduce (herramientas lentas, KB viva, límites de tasa). (3) Poco poder estadístico: una caída pequeña no es detectable con tu n. Y quizá (4) la regresión está en lo que el juez no mide (otro criterio).

</details>

<details>
<summary>5 · (Transferencia) Tu proveedor anuncia que retirará el modelo X y migrará el alias `default` a Y. ¿Qué plan de pruebas montas?</summary>

Replay del dataset de oro con X e Y (apareado) + huella de comportamiento sobre una muestra de tráfico real redactado; canario con enrutamiento determinista y SPRT con \(p_0,p_1\) acordados; guardarraíles de coste/latencia/tokens (Y puede ser más verboso); recalibrar el juez si usa Y; fijar la versión del modelo en la configuración y monitorizar `gen_ai.response.model` por si el proveedor cambia el modelo por debajo.

</details>

<details>
<summary>6 · (Transferencia) ¿Cómo aplicarías la huella de comportamiento a un agente de código que edita ficheros?</summary>

Con métricas del mismo estilo: nº de ficheros tocados, herramientas usadas (leer/editar/ejecutar tests), proporción de ejecuciones que corren tests antes de terminar, longitud de los diffs y tokens. Un cambio de prompt que reduce «ejecutar tests» es el equivalente a nuestro `no_tool_rate`. Todo ello sin juez y desde los spans `execute_tool`.

</details>

---
# 14 · Referencias

- OpenTelemetry, *Semantic conventions for generative AI* (repo `open-telemetry/semantic-conventions-genai`, estado *Development*): `gen_ai.agent.version`, `gen_ai.prompt.name`/`gen_ai.prompt.version`, evento `gen_ai.evaluation.result`.
- Wald, A. (1945). *Sequential Tests of Statistical Hypotheses.* Annals of Mathematical Statistics 16(2):117–186 — origen del SPRT.
- Johari, R., Koomen, P., Pekelis, L., Walsh, D. (2017). *Peeking at A/B Tests: Why It Matters, and What To Do About It.* KDD '17, pp. 1517–1525.
- Page, E. S. (1954). *Continuous Inspection Schemes.* Biometrika 41(1/2):100–115 — origen de CUSUM.
- McNemar, Q. (1947). *Note on the sampling error of the difference between correlated proportions or percentages.* Psychometrika 12(2):153–157.
- Kohavi, R., Tang, D., Xu, Y. (2020). *Trustworthy Online Controlled Experiments.* Cambridge University Press — experimentación online, poder, SRM, guardarraíles.
- Husain, H. (2024). *Your AI Product Needs Evals.* https://hamel.dev/blog/posts/evals/ — evals desde datos de producción.
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* arXiv:2306.05685.
- Beyer, B. et al. (2018). *The Site Reliability Workbook*, cap. 16 «Canarying Releases» — despliegues canario y su análisis. https://sre.google/workbook/canarying-releases/